# Choice-head ablation

Compare the flexible neural head with the positive symmetric head on the first-cycle Chilean sample. The result is exported by content to `choice_head_ablation.csv` and `choice_head_ablation.tex`, alongside each seed's accuracy.

**Inputs:** flexible-head metrics and geometric matrices in `outputs/ablation/flexible/`, generated by notebooks 04b and 14b from the supplied comparisons and coordinate fits. Positive symmetric metrics and matrices are generated by notebooks 02/04 and 14 in `outputs/diagnostics/` and `outputs/tables/`. These inputs are included, so this summary runs in a fresh kernel without first executing the training notebooks.

**Regenerate from observations:** use 02 and 04 with `REFIT=True` for the positive symmetric fits, and 04b with `REFIT=True` for the flexible fits. Then run 14 and 14b to recompute the geometry, followed by this notebook. With `REFIT=False`, the training notebooks reload supplied weights and reevaluate choices. No external benchmark table is needed to produce the ablation inputs.

Both specifications use 47,574 participants, 90 proposals, 2,650,103 training comparisons and 320,244 held-out comparisons. Training uses Adamax at learning rate 0.01, batch size 64, and the final of 20 epochs. Seeds are 101, 201, 301, 401 and 501. The heads contain 20 and 105 choice parameters, in addition to 95,328 embedding parameters. This comparison changes the entire choice function, not one constraint in isolation.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

# Run from the repository root or from notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
FLEXIBLE = ROOT / "outputs" / "ablation" / "flexible"
TABLES = ROOT / "outputs" / "tables"
DIAGNOSTICS = ROOT / "outputs" / "diagnostics"
OUTPUT = ROOT / "outputs" / "manuscript_tables"
SEEDS = [101, 201, 301, 401, 501]
MAIN_SEED = 101
FIT_NAMES = [f"S{i}" for i in range(len(SEEDS))]

paths = {
    "Flexible": {
        "metrics": FLEXIBLE / "initialization_metrics.csv",
        "participants": FLEXIBLE / "geometry_participants_pearson.csv",
        "proposals": FLEXIBLE / "geometry_proposals_pearson.csv",
        "pc1": FLEXIBLE / "geometry_pca_spearman.csv",
    },
    "Positive symmetric": {
        "metrics": DIAGNOSTICS / "Chile_initialization_metrics.csv",
        "participants": TABLES / "geometry_participants_pearson.csv",
        "proposals": TABLES / "geometry_proposals_pearson.csv",
        "pc1": TABLES / "geometry_pca_spearman.csv",
    },
}
missing = [str(p.relative_to(ROOT)) for group in paths.values() for p in group.values() if not p.is_file()]
if missing:
    raise FileNotFoundError("Restore the supplied inputs before running: " + ", ".join(missing))

## Held-out prediction across all five initializations

Accuracies below are percentages. The paired difference is flexible minus positive symmetric, in percentage points. The reported mean gives each initialization equal weight.

In [2]:
metrics = {}
for head, group in paths.items():
    frame = pd.read_csv(group["metrics"])
    required = {"seed", "test_loss_all_records", "test_accuracy_all_records"}
    if not required.issubset(frame.columns):
        raise ValueError(f"Incomplete metrics for {head}")
    if frame["seed"].duplicated().any() or set(frame["seed"]) != set(SEEDS):
        raise ValueError(f"Unexpected or duplicated seeds for {head}")
    frame = frame.set_index("seed").loc[SEEDS]
    if not np.isfinite(frame[list(required - {"seed"})].to_numpy()).all():
        raise ValueError(f"Non-finite metrics for {head}")
    if not frame["test_accuracy_all_records"].between(0, 1).all():
        raise ValueError(f"Accuracy outside [0, 1] for {head}")
    metrics[head] = frame

by_seed = pd.DataFrame({head: 100 * frame["test_accuracy_all_records"] for head, frame in metrics.items()})
by_seed["Difference (percentage points)"] = by_seed["Flexible"] - by_seed["Positive symmetric"]
with pd.option_context("display.float_format", "{:.4f}".format):
    display(by_seed)
display(pd.DataFrame({"Mean accuracy (%)": by_seed.iloc[:, :2].mean(),
                      "Minimum (%)": by_seed.iloc[:, :2].min(),
                      "Maximum (%)": by_seed.iloc[:, :2].max()}).round(4))
print(f"Mean paired difference: {by_seed.iloc[:, 2].mean():.4f} percentage points")

,Flexible,Positive symmetric,Difference (percentage points)
seed,,,
101,69.4517,68.0987,1.3530
201,68.0712,68.4375,-0.3663
301,68.6274,68.6483,-0.0209
401,68.5871,68.2242,0.3628
501,68.7913,68.5374,0.2539


,Mean accuracy (%),Minimum (%),Maximum (%)
Flexible,68.7057,68.0712,69.4517
Positive symmetric,68.3892,68.0987,68.6483


Mean paired difference: 0.3165 percentage points


## Correspondence of the learned geometry

Each correlation matrix covers S0–S4, corresponding in order to the five seeds listed above. Only its strict upper triangle is summarized. Pearson correlations of distances retain their signs. For PC1 we take the absolute value **for each fit pair before calculating the range**, because the projection direction is arbitrary. These ranges are descriptive, not confidence intervals.

In [3]:
geometry = {}
upper = np.triu_indices(len(SEEDS), k=1)
for head, group in paths.items():
    geometry[head] = {}
    for quantity in ("participants", "proposals", "pc1"):
        frame = pd.read_csv(group[quantity], index_col=0)
        if list(frame.index) != FIT_NAMES or list(frame.columns) != FIT_NAMES:
            raise ValueError(f"Unexpected fit order: {head}, {quantity}")
        matrix = frame.to_numpy(dtype=float)
        if not np.isfinite(matrix).all() or np.abs(matrix).max() > 1 + 1e-12:
            raise ValueError(f"Invalid correlations: {head}, {quantity}")
        if not np.allclose(matrix, matrix.T, rtol=0, atol=1e-12) or not np.allclose(np.diag(matrix), 1):
            raise ValueError(f"Invalid correlation matrix: {head}, {quantity}")
        values = matrix[upper]
        if quantity == "pc1":
            values = np.abs(values)
        geometry[head][quantity] = (float(values.min()), float(values.max()))

geometry_rows = [{"Head": head, "Quantity": quantity, "Minimum": limits[0], "Maximum": limits[1]}
                 for head, quantities in geometry.items() for quantity, limits in quantities.items()]
display(pd.DataFrame(geometry_rows).round(6))

,Head,Quantity,Minimum,Maximum
0,Flexible,participants,0.170283,0.629164
1,Flexible,proposals,-0.122343,0.427980
2,Flexible,pc1,0.010037,0.833053
3,Positive symmetric,participants,0.579881,0.689794
4,Positive symmetric,proposals,0.501133,0.629081
5,Positive symmetric,pc1,0.880589,0.912056


## Ablation table

The main-fit row retains seed 101. The mean and range use all five initializations. Files are identified by content, independently of the supplementary-table number assigned in a manuscript.

In [4]:
rows = [
    ("Choice parameters", "20", "105"),
    ("Main-fit accuracy (seed 101, %)", *[f"{by_seed.loc[MAIN_SEED, h]:.2f}" for h in paths]),
    ("Mean accuracy across five seeds (%)", *[f"{by_seed[h].mean():.2f}" for h in paths]),
    ("Accuracy range across five seeds (%)", *[f"{by_seed[h].min():.2f}--{by_seed[h].max():.2f}" for h in paths]),
]
for quantity, label in [("participants", "Participant distances (Pearson)"),
                        ("proposals", "Proposal distances (Pearson)"),
                        ("pc1", "Participant PC1 (absolute Spearman)")]:
    rows.append((label, *[f"{geometry[h][quantity][0]:.2f}--{geometry[h][quantity][1]:.2f}" for h in paths]))
summary = pd.DataFrame(rows, columns=["Quantity", "Flexible head", "Positive symmetric head"])
display(summary)

OUTPUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT / "choice_head_ablation.csv", index=False)
by_seed.to_csv(OUTPUT / "choice_head_ablation_by_seed.csv")
# Explicit escaping keeps percent signs and signed correlation ranges valid in LaTeX.
escape = lambda text: str(text).replace("%", r"\%")
lines = [r"\begin{tabular}{p{7.3cm}cc}", r"\toprule",
         r"Quantity & Flexible head & Positive symmetric head \\", r"\midrule"]
lines += [" & ".join(escape(value) for value in row) + r" \\" for row in rows]
lines += [r"\bottomrule", r"\end{tabular}"]
(OUTPUT / "choice_head_ablation.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Saved outputs/manuscript_tables/choice_head_ablation.{csv,tex}")

,Quantity,Flexible head,Positive symmetric head
0,Choice parameters,20,105
1,"Main-fit accuracy (seed 101, %)",69.45,68.10
2,Mean accuracy across five seeds (%),68.71,68.39
3,Accuracy range across five seeds (%),68.07--69.45,68.10--68.65
4,Participant distances (Pearson),0.17--0.63,0.58--0.69
5,Proposal distances (Pearson),-0.12--0.43,0.50--0.63
6,Participant PC1 (absolute Spearman),0.01--0.83,0.88--0.91


Saved outputs/manuscript_tables/choice_head_ablation.{csv,tex}


The comparison reports two complete choice functions. It does not isolate positivity, symmetry or parameter count, and does not test the positive multiplier against a constant multiplier.